# Double Pendulum: Transformer Training & Belief Probing on Kaggle GPU

**Domain & Mission**:
This notebook trains a causal transformer on the observation stream of a chaotic, Lagrangian **Double Pendulum** driven by a 4-mood Mess-4 Markov chain. We investigate whether the model linearly recovers the hidden Markovian predictive belief state $P(\text{next mood} \mid \text{history})$ and physical angular velocities $(\omega_1, \omega_2)$ from its residual streams.

### Key Pipeline Features:
1. **Directly Deployable on Kaggle**: Automatically handles repository cloning, environment discovery, and GPU selection.
2. **Double Pendulum Specialization**: Uses screened contractive damping ($\gamma_1 = \gamma_2 = 2.5$), 2D mixed-radix token grid ($30 \times 30 = 900$ tokens), and 4 balanced cardinal kicks (gain 2.0).
3. **On-the-Fly Streaming**: Zero disk I/O bottleneck; trajectories are generated on-the-fly during training.
4. **Log-Spaced Checkpointing & Emergence Probing**: Saves checkpoints along a geometric schedule to evaluate how belief representations emerge over training tokens.
5. **Direct Download**: Zips all checkpoints, metrics, and plots for easy download from Kaggle.


In [ ]:
# --- 1. KAGGLE & LOCAL REPOSITORY BOOTSTRAP ---
import os, sys, subprocess
from pathlib import Path

IS_KAGGLE = Path('/kaggle').exists()
REPO_URL = 'https://github.com/ManjotSingh08x/Belief-in-physics.git'
REPO_BRANCH = os.environ.get('BELIEF_REPO_BRANCH', 'double_pendulum_system')

def setup_repository():
    if not IS_KAGGLE:
        # Running locally
        candidate = Path.cwd()
        for p in [candidate, candidate.parent]:
            if (p / 'physics' / 'messk.py').exists():
                return p.resolve()
        raise FileNotFoundError("Local repo root not found. Please run inside Belief-in-physics.")

    # Running on Kaggle
    target = Path('/kaggle/working/Belief-in-physics')
    input_root = Path('/kaggle/input')
    
    # Check attached dataset first
    if input_root.exists():
        for marker in input_root.rglob('physics/messk.py'):
            repo_path = marker.parents[1]
            print(f"Found repository in Kaggle input: {repo_path}")
            return repo_path.resolve()

    # Clone repository if not found
    if not target.exists():
        print(f"Cloning branch '{REPO_BRANCH}' from {REPO_URL} ...")
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(target)], check=True)
    else:
        print(f"Repository already cloned at {target}")
        
    return target.resolve()

REPO_ROOT = setup_repository()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root set to: {REPO_ROOT}")

# Device selection
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using compute device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# --- 2. CONFIGURATION & IMPORTS ---
import time, json, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Import project modules
from physics.messk_configs import make_process, set_double_pendulum_bins
from physics.systems.double_pendulum import DoublePendulum
from models.transformer import ModelConfig, TinyTransformer
from models.train import TrainConfig, train

# --- RUN MODE CONTROLS ---
# Select 'FAST' (smoke test ~3 min), 'MODERATE' (50M tokens ~20 min), or 'FULL' (500M tokens ~3 hrs)
RUN_MODE = "FAST"

TOKEN_BUDGETS = {
    "FAST": 10_000_000,       # 10M tokens: quick validation of training & probe emergence
    "MODERATE": 50_000_000,   # 50M tokens: intermediate benchmark
    "FULL": 500_000_000       # 500M tokens: full paper-scale run
}

TOTAL_TOKENS = TOKEN_BUDGETS.get(RUN_MODE, 10_000_000)
BATCH_SIZE = 128
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
SEED = 0

# Model architecture (TinyTransformer 4x128)
EMBED_DIM = 128
NUM_LAYERS = 4
NUM_HEADS = 2
D_MLP = 4 * EMBED_DIM

# Checkpoint fractions along the training run
CHECKPOINT_FRACTIONS = (0.004, 0.01, 0.024, 0.06, 0.24, 0.50, 1.00)

# Output directory setup
OUTPUT_DIR = Path('/kaggle/working/outputs-double-pendulum') if IS_KAGGLE else REPO_ROOT / 'experiments' / 'outputs-double-pendulum'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR = OUTPUT_DIR / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Configuration: RUN_MODE={RUN_MODE} | TOTAL_TOKENS={TOTAL_TOKENS:,} | BATCH_SIZE={BATCH_SIZE}")
print(f"Model: {NUM_LAYERS} layers, d_model={EMBED_DIM}, d_mlp={D_MLP}, {NUM_HEADS} heads")
print(f"Outputs will be saved to: {OUTPUT_DIR}")


In [ ]:
# --- 3. DOUBLE PENDULUM PROCESS SETUP & VISUALIZATION ---
# Initialize the Double Pendulum process
proc = make_process("double_pendulum_mess4")
chain = proc.chain

print(f"System: double_pendulum_mess4")
print(f"Observation Vocabulary: {proc.n_obs} tokens (2D grid: 30 x 30)")
print(f"Observation Ranges: {proc.obs_ranges}")
print(f"Trajectory Length: {proc.seq_len} tokens (m={proc.m} cycles, n={proc.n_steps} steps/cycle)")
print(f"Actions Matrix (gain 2.0 on joint 1):\n{np.round(proc.actions, 3)}")

# Generate a diagnostic batch
rng = np.random.default_rng(SEED)
diag_batch = proc.sample_batch(rng, 8)
tokens = diag_batch["tokens"]        # (8, 160)
obs = diag_batch["observable"]       # (8, 160, 2)
beliefs = diag_batch["beliefs"]      # (8, 160, 4)
letters = diag_batch["letters"]      # (8, 16)

print(f"\nBatch shapes: tokens={tokens.shape}, obs={obs.shape}, beliefs={beliefs.shape}, letters={letters.shape}")

# Visualise diagnostic trajectory
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# Panel A: Angular trajectory (theta1, theta2)
t_steps = np.arange(proc.seq_len) * proc.dt
axes[0].plot(t_steps, obs[0, :, 0], label="$\theta_1$ (joint 1)", color="#2980b9", lw=1.5)
axes[0].plot(t_steps, obs[0, :, 1], label="$\theta_2$ (joint 2)", color="#e67e22", lw=1.5, ls="--")
for k_pos in range(0, proc.seq_len, proc.n_steps):
    axes[0].axvline(k_pos * proc.dt, color="gray", alpha=0.25, ls=":")
axes[0].set_title("Double Pendulum Joint Angles (Wrapped)")
axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("Angle (rad)")
axes[0].set_ylim(-np.pi, np.pi)
axes[0].legend(loc="upper right")
axes[0].grid(True, alpha=0.3)

# Panel B: 2D Token Grid Occupancy (30x30)
b1, b2 = proc.obs_bins
recon = proc.undiscretise(tokens[0])
h, xedges, yedges = np.histogram2d(recon[:, 0], recon[:, 1], bins=(b1, b2), range=[[-np.pi, np.pi], [-np.pi, np.pi]])
im = axes[1].imshow(h.T, origin="lower", extent=[-np.pi, np.pi, -np.pi, np.pi], cmap="Blues", aspect="equal")
axes[1].plot(recon[:, 0], recon[:, 1], color="#e74c3c", lw=1.0, alpha=0.7, label="Trajectory")
axes[1].scatter([recon[0, 0]], [recon[0, 1]], color="green", s=40, zorder=5, label="Start")
axes[1].set_title(f"2D Token Grid ({b1}x{b2} = {proc.n_obs} cells)")
axes[1].set_xlabel("$\theta_1$ (rad)")
axes[1].set_ylabel("$\theta_2$ (rad)")
axes[1].legend(loc="upper right", fontsize=8)

# Panel C: Hidden Predictive Beliefs
for mood_idx in range(4):
    axes[2].plot(t_steps, beliefs[0, :, mood_idx], label=f"P(mood {mood_idx})", lw=1.5)
axes[2].set_title("Ground-Truth Hidden Belief $P(\text{mood}_t \mid \text{history})$")
axes[2].set_xlabel("Time (s)")
axes[2].set_ylabel("Belief Probability")
axes[2].set_ylim(-0.05, 1.05)
axes[2].legend(loc="upper right", fontsize=8)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "double_pendulum_diagnostics.png", dpi=150)
plt.show()


In [ ]:
# --- 4. MODEL INITIALIZATION & TRAINING ---
# Create Model Config
model_cfg = ModelConfig(
    vocab_size=proc.n_obs,
    n_ctx=proc.seq_len,
    n_layers=NUM_LAYERS,
    n_heads=NUM_HEADS,
    d_model=EMBED_DIM,
    d_mlp=D_MLP,
    seed=SEED
)

# Initialize untrained baseline model (0% control)
random_init = TinyTransformer(model_cfg)
random_init_path = OUTPUT_DIR / "double_pendulum_random_init.pt"
torch.save(random_init.state_dict(), random_init_path)

# Initialize active training model
model = TinyTransformer(model_cfg)
model.load_state_dict(random_init.state_dict())
model.to(DEVICE)

# Checkpointing schedule
checkpoint_steps = tuple(sorted({int(f * TOTAL_TOKENS) for f in CHECKPOINT_FRACTIONS}))
saved_checkpoints = {0: random_init_path}

def checkpoint_callback(tokens_seen: int, current_model: TinyTransformer):
    ckpt_path = CKPT_DIR / f"double_pendulum_{tokens_seen}.pt"
    torch.save(current_model.state_dict(), ckpt_path)
    saved_checkpoints[tokens_seen] = ckpt_path
    print(f"Saved checkpoint at {tokens_seen:,} tokens -> {ckpt_path.name}")

print(f"Checkpoint schedule (tokens): {checkpoint_steps}")
print(f"Uniform token loss baseline: ln({proc.n_obs}) = {math.log(proc.n_obs):.4f} nats")

# Training execution
t0 = time.perf_counter()
train_config = TrainConfig(
    total_tokens=TOTAL_TOKENS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    seed=SEED,
    log_every=25,
    checkpoint_at=checkpoint_steps
)

print(f"\nStarting streaming training on {DEVICE.upper()} ...")
training_report = train(
    model,
    lambda rng_instance, n: proc.sample_batch(rng_instance, n)["tokens"],
    seq_len=proc.seq_len,
    config=train_config,
    device=DEVICE,
    on_checkpoint=checkpoint_callback
)

# Save final trained weights
final_path = OUTPUT_DIR / "double_pendulum_final.pt"
torch.save(model.state_dict(), final_path)
saved_checkpoints[TOTAL_TOKENS] = final_path
training_time = time.perf_counter() - t0

print(f"\nTraining complete in {training_time:.1f}s ({training_time/60:.2f} min)!")
print(f"Final evaluation loss: {training_report['final']['eval_loss']:.4f} (baseline: {math.log(proc.n_obs):.4f})")


In [ ]:
# --- 5. TRAINING LOSS CURVE ---
history = training_report["history"]
tokens_history = [entry["tokens"] for entry in history]
loss_history = [entry["train_loss"] for entry in history]
eval_tokens = [entry["tokens"] for entry in history if "eval_loss" in entry]
eval_losses = [entry["eval_loss"] for entry in history if "eval_loss" in entry]

plt.figure(figsize=(9, 4.5))
plt.plot(tokens_history, loss_history, label="Training Loss", color="#2980b9", lw=1.5, alpha=0.85)
if eval_losses:
    plt.scatter(eval_tokens, eval_losses, color="#e74c3c", s=30, zorder=5, label="Eval Loss")
plt.axhline(math.log(proc.n_obs), color="#e67e22", ls="--", lw=1.2, label=f"Uniform Baseline ({math.log(proc.n_obs):.3f})")

for ckpt_tok in checkpoint_steps:
    plt.axvline(ckpt_tok, color="gray", alpha=0.3, ls=":")

plt.title("Double Pendulum Next-Token Prediction Loss")
plt.xlabel("Tokens Seen")
plt.ylabel("Cross-Entropy Loss (nats)")
plt.legend(loc="upper right")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "training_loss_curve.png", dpi=150)
plt.show()


In [ ]:
# --- 6. LINEAR PROBING OF RESIDUAL STREAMS ---
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import KFold
from models.analysis import residual_streams_batched

print("Generating held-out evaluation dataset for linear probing ...")
eval_rng = np.random.default_rng(20260925)
N_EVAL_TRAJECTORIES = 256
eval_batch = proc.sample_batch(eval_rng, N_EVAL_TRAJECTORIES)

eval_tokens = torch.from_numpy(eval_batch["tokens"]).long()
eval_beliefs = eval_batch["beliefs"]      # (N, 160, 4)
eval_metrics = eval_batch["metric"]       # (N, 160, 2) joint velocities (omega1, omega2)

# Flatten targets across all sequence positions (N * 160, targets)
y_belief = eval_beliefs.reshape(-1, 4)
y_metric = eval_metrics.reshape(-1, 2)

print(f"Evaluation targets shape: belief={y_belief.shape}, metric={y_metric.shape}")

# Function to run ridge regression probe with cross-validation
def probe_representations(features, targets, cv_splits=5, alpha=10.0):
    kf = KFold(n_splits=cv_splits, shuffle=True, random_state=42)
    oof_preds = np.zeros_like(targets)
    
    for train_idx, val_idx in kf.split(features):
        clf = Ridge(alpha=alpha)
        clf.fit(features[train_idx], targets[train_idx])
        oof_preds[val_idx] = clf.predict(features[val_idx])
        
    return r2_score(targets, oof_preds)

probe_results = []
checkpoint_keys = sorted(saved_checkpoints.keys())

print(f"\nProbing representations across {len(checkpoint_keys)} checkpoints and {NUM_LAYERS+1} layers ...")
for tokens_seen in tqdm(checkpoint_keys, desc="Checkpoints"):
    ckpt_file = saved_checkpoints[tokens_seen]
    probe_model = TinyTransformer(model_cfg)
    probe_model.load_state_dict(torch.load(ckpt_file, map_location=DEVICE))
    probe_model.to(DEVICE)
    probe_model.eval()
    
    # Extract residual streams: shape (n_layers + 1, batch_size, seq_len, d_model)
    with torch.no_grad():
        activations = residual_streams_batched(probe_model, eval_tokens, batch_size=64, device=DEVICE)
        
    for layer_idx, layer_acts in enumerate(activations):
        layer_name = "Embedding" if layer_idx == 0 else f"Layer {layer_idx}"
        feats = layer_acts.reshape(-1, EMBED_DIM)
        
        r2_b = probe_representations(feats, y_belief)
        r2_m = probe_representations(feats, y_metric)
        
        probe_results.append({
            "tokens_seen": tokens_seen,
            "layer_idx": layer_idx,
            "layer_name": layer_name,
            "belief_r2": float(r2_b),
            "metric_r2": float(r2_m)
        })

df_probes = pd.DataFrame(probe_results)
df_probes.to_csv(OUTPUT_DIR / "probe_results.csv", index=False)
print(f"\nProbing complete! Results saved to {OUTPUT_DIR / 'probe_results.csv'}")


In [ ]:
# --- 7. EMERGENCE & REPRESENTATION VISUALIZATION ---
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Panel A: Emergence Curves (Belief R^2 vs Tokens Seen)
for layer_idx in range(NUM_LAYERS + 1):
    sub = df_probes[df_probes["layer_idx"] == layer_idx]
    layer_name = sub["layer_name"].iloc[0]
    axes[0].plot(sub["tokens_seen"], sub["belief_r2"], marker="o", label=layer_name, lw=1.8)

axes[0].set_title("Belief State Recovery ($R^2$) Emergence Over Training")
axes[0].set_xlabel("Tokens Seen")
axes[0].set_ylabel("Belief Probe $R^2$")
axes[0].set_ylim(-0.1, 1.0)
axes[0].grid(True, alpha=0.3)
axes[0].legend(loc="lower right")

# Panel B: Final Layer Profile (Belief vs Physical Metrics)
final_probes = df_probes[df_probes["tokens_seen"] == max(checkpoint_keys)]
x_layers = np.arange(NUM_LAYERS + 1)
w = 0.35

axes[1].bar(x_layers - w/2, final_probes["belief_r2"], width=w, label="Hidden Belief ($R^2$)", color="#3498db")
axes[1].bar(x_layers + w/2, final_probes["metric_r2"], width=w, label="Velocity ($\omega_1, \omega_2$) ($R^2$)", color="#e67e22")

axes[1].set_xticks(x_layers)
axes[1].set_xticklabels(final_probes["layer_name"])
axes[1].set_title(f"Final Model Representation Readout by Depth ({max(checkpoint_keys):,} tokens)")
axes[1].set_ylabel("Probe $R^2$ Score")
axes[1].set_ylim(0.0, 1.0)
axes[1].grid(True, alpha=0.3, axis="y")
axes[1].legend(loc="upper left")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "belief_emergence_and_readout.png", dpi=150)
plt.show()

# Display summary comparison table
print("=== REPRESENTATION READOUT COMPARISON ===")
summary_table = df_probes[df_probes["tokens_seen"].isin([0, max(checkpoint_keys)])].pivot(
    index="layer_name", columns="tokens_seen", values=["belief_r2", "metric_r2"]
)
display(summary_table)


In [ ]:
# --- 8. PACKAGE ARTIFACTS FOR KAGGLE DOWNLOAD ---
import shutil
from IPython.display import FileLink, HTML, display

zip_target = Path('/kaggle/working/double_pendulum_results') if IS_KAGGLE else REPO_ROOT / 'double_pendulum_results'
zip_file = shutil.make_archive(str(zip_target), 'zip', str(OUTPUT_DIR))

print(f"\nAll checkpoints, plots, and CSV reports packaged successfully into:\n{zip_file}")
file_size_mb = Path(zip_file).stat().st_size / (1024 * 1024)
print(f"Archive size: {file_size_mb:.2f} MB")

if IS_KAGGLE:
    display(HTML(f'<h3><a href="/kaggle/working/double_pendulum_results.zip" download>📥 Click here to download double_pendulum_results.zip</a></h3>'))
else:
    print(f"Local archive ready at: {zip_file}")
